# Setup:

In [2]:
import torch
from transformers import AdamW, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, get_cosine_schedule_with_warmup
from tqdm import tqdm
import os
from datasets import Dataset
import random
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
import numpy as np
from collections import defaultdict
from tqdm import tqdm
import concurrent.futures
from functools import partial
from itertools import product

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
DATASET_ROOT = "../../CrossVul"
DATASET_ROOT = "../../Preprocessed/Rename"
DATASET_ROOT = "../../Preprocessed/NoRename"
DATASET_ROOTS = ["../../CrossVul", "../../Preprocessed/Rename", "../../Preprocessed/NoRename"]
ALLOWED_CWE_IDS = {"CWE-787", "CWE-79", "CWE-787", "CWE-89"}
LANGUAGES = ['c', 'cpp', 'cs', 'java', 'py', 'php']
SEED = 42
EPOCHS = 3

In [3]:
codeBERT = "microsoft/codebert-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(codeBERT, trust_remote_code=True)
print(device)

cuda


In [4]:
class FileAwareTrainer(Trainer):
    def __init__(self, *args, eval_dataset_filenames=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.eval_dataset_filenames = eval_dataset_filenames

    def evaluate(self, eval_dataset=None, **kwargs):
        output = super().evaluate(eval_dataset=eval_dataset, **kwargs)
        self._last_eval_preds = kwargs.get('preds', None)
        return output

    def predict(self, test_dataset, **kwargs):
        self.eval_dataset_filenames = test_dataset['filename']
        return super().predict(test_dataset, **kwargs)

# Data Preprocessing

In [5]:
def collect_files_for_cwe(cwe_id, dataset):
    samples = []
    for lang in LANGUAGES:
        lang_dir = os.path.join(dataset, cwe_id, lang)
        if not os.path.isdir(lang_dir):
            continue
        for filename in os.listdir(lang_dir):
            filepath = os.path.join(lang_dir, filename)
            if filename.endswith('.DS_Store'):
                continue
            label = 1 if "bad" in filename.lower() else 0
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                code = f.read()
            samples.append({
                "filename": filename,
                "code": code,
                "label": label
            })
    print(len(samples))
    return samples

def compute_file_metrics_builder(filenames, thresh):
    def compute_file_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        file_pred_chunks = defaultdict(list)
        file_label = {}

        for pred, label, fname in zip(preds, labels, filenames):
            file_pred_chunks[fname].append(pred)
            file_label[fname] = label

        final_preds, final_labels = [], []
        for fname in file_pred_chunks:
            final_labels.append(file_label[fname])
            vulnerable_chunks = sum(1 for pred in file_pred_chunks[fname] if pred == 1)
            if vulnerable_chunks / len(file_pred_chunks[fname]) >= thresh:
                final_preds.append(1)
            else:
                final_preds.append(0)

        precision, recall, f1, _ = precision_recall_fscore_support(final_labels, final_preds, average='macro')
        acc = accuracy_score(final_labels, final_preds)
        confusion = confusion_matrix(final_labels, final_preds).tolist()
        ch_confusion = confusion_matrix(labels, preds).tolist()
        print(f"file level: {confusion}")
        print(f"chunk level: {ch_confusion}")
        return {
            'accuracy': acc,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            "confusion_matrix": confusion
        }

    return compute_file_metrics

def tokenize_example(batch, max_length=512):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    filenames_list = []

    for code, label, filename in zip(batch["code"], batch["label"], batch["filename"]):
        tokens = tokenizer(code, return_attention_mask=True, truncation=False)
        input_ids = tokens["input_ids"]
        attention_mask = tokens["attention_mask"]

        for i in range(0, len(input_ids), max_length):
            chunk_ids = input_ids[i:i + max_length]
            chunk_mask = attention_mask[i:i + max_length]

            pad_len = max_length - len(chunk_ids)
            if pad_len > 0:
                chunk_ids += [tokenizer.pad_token_id] * pad_len
                chunk_mask += [0] * pad_len

            input_ids_list.append(chunk_ids)
            attention_mask_list.append(chunk_mask)
            labels_list.append(label)
            filenames_list.append(filename)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "label": labels_list,
        "filename": filenames_list
    }

# Model Finetuning:

In [8]:
import csv
import os
from transformers import EarlyStoppingCallback

EPOCHS_LIST = [3]
LEARNING_RATES = [2e-5]
WEIGHT_DECAYS = [0.01]
BATCH_SIZES = [8]
CHUNK_THRESHES = [0.1]
LAYERS_TO_UNFREEZE = [0]

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.0
)

cnt = 0
for dataset in DATASET_ROOTS:
    for cwe_id in ALLOWED_CWE_IDS:
        if cnt < 8:
            cnt += 1
            continue
        print(f"\n--- Grid Search for {cwe_id} ---")
        samples = collect_files_for_cwe(cwe_id, dataset)
        random.seed(SEED)
        random.shuffle(samples)
        raw_dataset = Dataset.from_list(samples)
        train_test = raw_dataset.train_test_split(test_size=0.2, seed=SEED)
        train_raw = train_test["train"]
        eval_raw = train_test["test"]
        tokenized_train = train_raw.map(tokenize_example, batched=True, remove_columns=["filename", "code"])
        tokenized_eval = eval_raw.map(tokenize_example, batched=True, remove_columns=["filename", "code"])
        tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label', 'filename'])
        tokenized_eval.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label', 'filename'])

        train_dataset = tokenized_train
        eval_dataset = tokenized_eval
        filenames = eval_dataset["filename"]

        log_path = f"./models/codebert_{cwe_id}/gridsearch_results.csv"
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        with open(log_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["epochs", "lr", "weight_decay", "batch_size", "unfrozen_layers", "chunk_thresh", "precision", "recall", "f1", "accuracy", "confusion_matrix"])
            
        best_f1 = -1
        best_dir = None

        for epochs in EPOCHS_LIST:
            for lr in LEARNING_RATES:
                for wd in WEIGHT_DECAYS:
                    for batch_size in BATCH_SIZES:
                        for unfrozen in LAYERS_TO_UNFREEZE:
                            for chunk_thresh in CHUNK_THRESHES:
                                print(f"\nRunning with epochs={epochs}, lr={lr}, wd={wd}, batch_size={batch_size}, ch_thresh={chunk_thresh}, unfrozen_layers={unfrozen}, dataset={dataset}")
                                model = AutoModelForSequenceClassification.from_pretrained(codeBERT, num_labels=2).to(device)

                                # roberta has 12 layers.
                                if (unfrozen == -1): #make all layers trainable
                                    for param in model.parameters():
                                        param.requires_grad = True
                                
                                else:
                                    if unfrozen >= 6:
                                        if hasattr(model.base_model, "embeddings"):
                                            for param in model.base_model.embeddings.parameters():
                                                param.requires_grad = True

                                    if hasattr(model.base_model, 'encoder'):
                                        encoder_layers = model.base_model.encoder.layer
                                        if isinstance(encoder_layers, torch.nn.ModuleList):
                                            for layer in encoder_layers[-unfrozen:]:
                                                for param in layer.parameters():
                                                    param.requires_grad = True

                                    for param in model.classifier.parameters():
                                        param.requires_grad = True

                                optimizer = AdamW(model.parameters(), lr=lr, weight_decay=wd)
                                num_train_steps = len(train_dataset) * epochs
                                warmup_steps = int(0.1 * num_train_steps)
                                scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, num_train_steps)

                                output_dir = f"./models/codebert_{cwe_id}/gridsearch/ep{epochs}_lr{lr}_wd{wd}_bs{batch_size}_uf{unfrozen}_ct{chunk_thresh}"
                                training_args = TrainingArguments(
                                    output_dir=output_dir,
                                    evaluation_strategy="epoch",
                                    learning_rate=lr,
                                    per_device_train_batch_size=batch_size,
                                    per_device_eval_batch_size=batch_size,
                                    num_train_epochs=epochs,
                                    weight_decay=wd,
                                    save_strategy="epoch",
                                    load_best_model_at_end=True,
                                    metric_for_best_model="eval_f1",
                                    greater_is_better=True,
                                    remove_unused_columns=False,
                                    logging_dir="./logs",
                                    logging_strategy="epoch",
                                    save_total_limit=1,
                                )

                                trainer = FileAwareTrainer(
                                    model=model,
                                    args=training_args,
                                    train_dataset=train_dataset,
                                    eval_dataset=eval_dataset,
                                    compute_metrics=compute_file_metrics_builder(filenames, chunk_thresh),
                                    optimizers=(optimizer, scheduler),
                                    callbacks=[early_stopping_callback]
                                )

                                trainer.train()
                                trainer.save_model(output_dir + "/final")

                                metrics = trainer.evaluate()
                                precision = metrics["eval_precision"]
                                recall = metrics["eval_recall"]
                                f1 = metrics["eval_f1"]
                                accuracy = metrics["eval_accuracy"]
                                confusion = metrics["eval_confusion_matrix"]

                                with open(log_path, "a", newline="") as f:
                                    writer = csv.writer(f)
                                    writer.writerow([epochs, lr, wd, batch_size, unfrozen, chunk_thresh, precision, recall, f1, accuracy, confusion])

                                if f1 > best_f1:
                                    best_f1 = f1
                                    best_dir = output_dir
                                    best_thresh = chunk_thresh
                                                            
        if best_dir is not None:
            os.system(f"cp -r {best_dir}/final ./models/codebert_{cwe_id}/best_model")
            print(f"\nBest model for {cwe_id} saved from: {best_dir} with F1={best_f1:.4f}")


--- Grid Search for CWE-89 ---
554


Map:   0%|          | 0/443 [00:00<?, ? examples/s]

Map:   0%|          | 0/111 [00:00<?, ? examples/s]


Running with epochs=3, lr=2e-05, wd=0.01, batch_size=8, ch_thresh=0.1, unfrozen_layers=0, dataset=../../Preprocessed/NoRename


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RuntimeError: CUDA error: an illegal memory access was encountered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
